In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
PROCESSED_DIR = Path("../data/processed")
CHECKPOINT_PATH = Path("../outputs/checkpoints/best_physics_unet_hybrid.pth")

BATCH_SIZE = 8

print("Processed exists:", PROCESSED_DIR.exists())
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Checkpoint path:", CHECKPOINT_PATH)

In [ ]:
class LandsatPatchDataset(Dataset):
    def __init__(self, processed_dir, split):
        self.input_dir = Path(processed_dir) / split / "inputs"
        self.target_dir = Path(processed_dir) / split / "targets"

        self.input_files = sorted(self.input_dir.glob("*.npy"))
        self.target_files = sorted(self.target_dir.glob("*.npy"))

        assert len(self.input_files) == len(self.target_files), "Input/target count mismatch"

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        inp = np.load(self.input_files[idx]).astype(np.float32)
        tgt = np.load(self.target_files[idx]).astype(np.float32)

        inp = torch.from_numpy(inp).permute(2, 0, 1)
        tgt = torch.from_numpy(tgt).permute(2, 0, 1)

        return inp, tgt

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class PhysicsUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3):
        super().__init__()

        self.down1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.down4 = DoubleConv(128, 256)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up4 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.conv4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.conv1 = DoubleConv(64, 32)

        self.out = nn.Conv2d(32, out_channels, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool1(d1))
        d3 = self.down3(self.pool2(d2))
        d4 = self.down4(self.pool3(d3))

        bn = self.bottleneck(self.pool4(d4))

        u4 = self.up4(bn)
        u4 = self.conv4(torch.cat([u4, d4], dim=1))

        u3 = self.up3(u4)
        u3 = self.conv3(torch.cat([u3, d3], dim=1))

        u2 = self.up2(u3)
        u2 = self.conv2(torch.cat([u2, d2], dim=1))

        u1 = self.up1(u2)
        u1 = self.conv1(torch.cat([u1, d1], dim=1))

        return self.activation(self.out(u1))

In [ ]:
model = PhysicsUNet(in_channels=5, out_channels=3).to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded model:", checkpoint.get("model_name", "PhysicsUNet_HybridLoss"))
print("Loaded epoch:", checkpoint["epoch"] + 1)
print("Best val loss:", checkpoint["val_loss"])

In [ ]:
test_dataset = LandsatPatchDataset(PROCESSED_DIR, "test")

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

In [ ]:
psnr_scores = []
ssim_scores = []

with torch.no_grad():
    for inputs, targets in tqdm(test_loader):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        preds = model(inputs)

        preds_np = preds.cpu().numpy()
        targets_np = targets.cpu().numpy()

        for i in range(preds_np.shape[0]):
            pred_img = np.transpose(preds_np[i], (1, 2, 0))
            target_img = np.transpose(targets_np[i], (1, 2, 0))

            pred_img = np.clip(pred_img, 0, 1)
            target_img = np.clip(target_img, 0, 1)

            psnr = peak_signal_noise_ratio(
                target_img,
                pred_img,
                data_range=1.0
            )

            ssim = structural_similarity(
                target_img,
                pred_img,
                channel_axis=-1,
                data_range=1.0
            )

            psnr_scores.append(psnr)
            ssim_scores.append(ssim)

avg_psnr = float(np.mean(psnr_scores))
avg_ssim = float(np.mean(ssim_scores))

print("Average PSNR:", avg_psnr)
print("Average SSIM:", avg_ssim)

In [ ]:
def show_prediction(idx=None):
    if idx is None:
        idx = np.random.randint(0, len(test_dataset))

    inp, tgt = test_dataset[idx]

    with torch.no_grad():
        pred = model(inp.unsqueeze(0).to(device)).squeeze(0).cpu()

    inp_np = inp.permute(1, 2, 0).numpy()
    tgt_np = tgt.permute(1, 2, 0).numpy()
    pred_np = pred.permute(1, 2, 0).numpy()

    thermal = inp_np[:, :, 0]
    ndvi = inp_np[:, :, 1]
    ndwi = inp_np[:, :, 2]
    ndbi = inp_np[:, :, 3]

    error = np.abs(tgt_np - pred_np).mean(axis=-1)

    plt.figure(figsize=(22, 5))

    plt.subplot(1, 6, 1)
    plt.imshow(thermal, cmap="gray")
    plt.title("Input Thermal")
    plt.axis("off")

    plt.subplot(1, 6, 2)
    plt.imshow(ndvi, cmap="gray")
    plt.title("NDVI")
    plt.axis("off")

    plt.subplot(1, 6, 3)
    plt.imshow(np.clip(pred_np, 0, 1))
    plt.title("Predicted RGB")
    plt.axis("off")

    plt.subplot(1, 6, 4)
    plt.imshow(np.clip(tgt_np, 0, 1))
    plt.title("Ground Truth RGB")
    plt.axis("off")

    plt.subplot(1, 6, 5)
    plt.imshow(error, cmap="hot")
    plt.title("Error Map")
    plt.axis("off")

    plt.subplot(1, 6, 6)
    diff_rgb = np.clip(np.abs(tgt_np - pred_np) * 4, 0, 1)
    plt.imshow(diff_rgb)
    plt.title("RGB Difference x4")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
for _ in range(5):
    show_prediction()

In [ ]:
import json

metrics = {
    "model": "PhysicsUNet_HybridLoss",
    "checkpoint": str(CHECKPOINT_PATH),
    "average_psnr": avg_psnr,
    "average_ssim": avg_ssim,
    "test_samples": len(test_dataset),
}

metrics_path = Path("../outputs/metrics")
metrics_path.mkdir(parents=True, exist_ok=True)

with open(metrics_path / "hybrid_physics_unet_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved metrics to:", metrics_path / "hybrid_physics_unet_metrics.json")